In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import sys

project_root = Path().cwd().parents[0]
sys.path.insert(0, str(project_root))

from scripts.explanatory_data_analysis import (
    fetch_dataset_and_turn_into_dataframe_and_convert_column_names_to_lowercase,
    dataset_quality_inspection,
)
from scripts.data_preprocessing import (
    data_clues_and_target,
    data_train_test_split,
    identify_feature_groups,
    create_numerical_imputer,
    create_categorical_imputer,
    create_ordinal_encoder,
    create_binary_encoder,
    create_nominal_encoder,
    create_social_media_screen_time_transformer,
    create_numerical_scaler,
    create_numerical_pipeline,
    create_ordinal_pipeline,
    create_nominal_pipeline,
    create_binary_pipeline,
    create_preprocessor,
)

from scripts.model_training import (
    create_model_pipeline
)

from sklearn.linear_model import LogisticRegression

raw_data_path = project_root / "data" / "raw" / "student_mental_health.csv"

raw_df = fetch_dataset_and_turn_into_dataframe_and_convert_column_names_to_lowercase(
    raw_data_path
)

clues_and_target = data_clues_and_target(raw_df)

# DataFrame Model Clues
X = clues_and_target["X"]
# DataFrame Model Target
y = clues_and_target["y"]


display(X.columns)
display(y)

================<class 'sklearn.linear_model._logistic.LogisticRegression'>


Index(['age', 'gender', 'year_of_study', 'sleep_hours', 'study_hours_per_week',
       'gpa', 'social_media_hours', 'screen_time_hours', 'physical_activity',
       'financial_stress', 'family_support', 'attendance_rate',
       'part_time_job', 'diet_quality', 'previous_mental_health_history'],
      dtype='str')

0         High
1       Medium
2       Medium
3         High
4       Medium
         ...  
1995    Medium
1996      High
1997      High
1998       Low
1999    Medium
Name: mental_health_risk, Length: 2000, dtype: str

In [ ]:
# Data Train Test Split
X_train, X_test, y_train, y_test = data_train_test_split(raw_df)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape[0])
print(y_test.shape[0])

print("=" * 30)

# Make sure the traget distribution is balanced
print("TRAIN TARGET DISTRIBUTION")
print(y_train.value_counts(normalize=True) * 100)
print("=" * 30)
print("TEST TARGET DISTRIBUTION")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
# Identify data feature groups
feature_groups = identify_feature_groups(X_train)
numerical_group = feature_groups["numerical_group"]
ordinal_categorical_group = feature_groups["ordinal_categorical_group"]
nominal_categorical_group = feature_groups["nominal_categorical_group"]
binary_categorical_group = feature_groups["binary_categorical_group"]

print(f"NUMERICAL GROUP: {numerical_group}")
print("=" * 30)
print(f"ORDINAL CATEGORICAL GROUP: {ordinal_categorical_group}")
print("=" * 30)
print(f"NOMINAL CATEGORICAL GROUP: {nominal_categorical_group}")
print("=" * 30)
print(f"BINARY CATEGOIRICAL GROUP: {binary_categorical_group}")

In [ ]:
# Initialize numerical pipeline
numerical_pipeline = create_numerical_pipeline(
    imputer=create_numerical_imputer(),
    transformer=create_social_media_screen_time_transformer(),
    scaler=create_numerical_scaler(),
)

# Initialize ordinal pipeline
ordinal_pipeline = create_ordinal_pipeline(
    imputer=create_categorical_imputer(), encoder=create_ordinal_encoder()
)

# Initialize nominal pipeline
nominal_pipeline = create_nominal_pipeline(
    imputer=create_categorical_imputer(), encoder=create_nominal_encoder()
)

# Initialize binary pipeline
binary_pipeline = create_binary_pipeline(
    imputer=create_categorical_imputer(), encoder=create_binary_encoder()
)

# Initialize preprocessor (ColumnTransformer)
preprocessor = create_preprocessor(
    numerical_pipeline=numerical_pipeline,
    ordinal_pipeline=ordinal_pipeline,
    nominal_pipeline=nominal_pipeline,
    binary_pipeline=binary_pipeline,
    numerical_group=numerical_group,
    ordinal_categorical_group=ordinal_categorical_group,
    nominal_categorical_group=nominal_categorical_group,
    binary_categorical_group=binary_categorical_group,
)

In [ ]:
# Initialize LogisticRegression pipeline 
logistic_regression_model = LogisticRegression(max_iter=1000, random_state=42)

logistic_regression_pipeline = create_model_pipeline(preprocessor=preprocessor, classifier=logistic_regression_model)